# Portfolio Optimization for SFO: from Closed Form to Simulation

A step-by-step implementation of the leading allocation example from A. Meucci, *Risk and Asset Allocation*, Section 6.1, on three real ETFs: **DFUS** (US equity), **DFAI** (international developed), **DFAE** (emerging markets).

## Scope: a minimum viable model
The plan: start from the book's fully analytical solution, then replace one ingredient at a time with its numerical counterpart, requiring at each step that the new answer reproduces the previous one wherever they overlap.

1. **Analytical solution.** Assumptions: horizon prices are (approximately) normal, utility is exponential (CARA), the index of satisfaction is the certainty-equivalent (CE), and risk is monitored through a VaR budget. The optimal allocation is the closed form (6.39).
2. **Convex optimization.** The same problem written as a second-order cone program in `cvxpy` — must match (6.39) to solver tolerance.
3. **VaR → CVaR.** The quantile constraint is replaced by the tail average (expected shortfall). Still closed-form under normality — and, unlike VaR, CVaR remains convex on simulated scenarios, which is what makes step 4 possible.
4. **Exact lognormal market.** Normal invariants actually imply *lognormal* horizon prices, and the portfolio value — a sum of correlated lognormals — has no parametric distribution at all. We switch to Monte Carlo scenarios (Sample Average Approximation) and re-solve, using steps 1–3 as validation anchors.

Fixed throughout: initial wealth $w$ = USD 100M, risk tolerance $\zeta$ = USD 10M, confidence level $c$ = 95%, investment horizon $\tau$ = 1 week.

## Scope: a practical model

This notebook is deliberately a **minimum viable model**: at every stage of the pipeline we take the simplest defensible choice — three liquid equity ETFs, i.i.d. normal invariants, plain sample moments, trivial projection, CARA utility, no transaction costs. The point of building the full MVM first is methodological: once the end-to-end pipeline is validated (each numerical method checked against a closed form wherever they overlap), every later refinement replaces exactly **one** block while the rest of the machinery — and its checkpoints — stays fixed.

Planned extensions, by pipeline stage:

**Invariants (Ch. 3.1)**
- Add **bonds** (Ch. 3.1.2): the equity recipe does not transfer — a bond's time-to-maturity shrinks, so raw returns are not time-homogeneous; the invariants are changes in yield to maturity at a *fixed* time-to-maturity.
- Add **real estate in Abu Dhabi** (open research question): appraisal-based and illiquid price series violate the i.i.d. invariant construction (smoothing, stale prices); requires its own study before it can enter the model.

**Estimation (Ch. 4)**
- **Shrinkage estimators** of the moments (Ch. 4.4): trade a little bias for a large variance reduction — especially important for the mean vector, which drives the allocation and is the noisiest input.
- **Robust M-estimators** (Ch. 4.5): reduce the influence of outliers on $\hat\mu, \hat\Sigma$.
- **M-estimators combined with shrinkage** (recent literature).

**Projection (Ch. 3.2)**
- **1 Year horizon**
- **Cyclostationary model**: relax "identically distributed across the calendar" — allow periodic (seasonal) structure in the invariant distribution instead of the plain $\tau/\tilde\tau$ scaling.

**Views / beliefs (Part III of the Meucci's)**
- Incorporate subjective beliefs about the future alongside the historical estimates. Two book routes: **Bayesian estimation** (Ch. 7 — prior + data $\to$ posterior distribution of the invariants) and **Black–Litterman** (§9.2 — equilibrium prior blended with views on expected returns). A third, scenario-native route from Meucci's later work is **entropy pooling** ("Fully Flexible Views", 2008): keep the simulated scenarios, reweight their probabilities to satisfy the views with minimal distortion — it plugs directly into the scenario/SAA framework of step 4, which already supports non-uniform scenario weights.

**Utility (Ch 5.4)**
- Move from CARA to the **HARA family**, which nests exponential (used here), power/CRRA and quadratic utility — allowing risk aversion to depend on wealth.

**Optimization**
- Add **transaction costs** (6.14), turning the budget constraint into (6.23) — the cost term $k'|\tilde\alpha - \alpha|$ is convex piecewise-linear, so it slots directly into the convex formulation.

In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np
import cvxpy as cp
import pandas as pd
from emfin_capstone.toolbox import price, returns, normal_market
from emfin_capstone import data_types as dt
from datetime import date
from plotly import graph_objects as go
import plotly.express as px
from scipy import special

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Invariants: raw data (Meucci Ch. 3.1)

Market invariants — quantities that repeat i.i.d. across time — are, for equities, the **compounded (log) returns**. Construction choices:

- **Dividend-adjusted** close prices (total return). These funds distribute ≈2%/yr; unadjusted prices would bias the means down.
- **Weekly** frequency, **Wednesday-to-Wednesday** closes, to stay clear of weekend/Monday effects.
- A return is kept only when two consecutive Wednesday closes are exactly 7 calendar days apart. This drops the weeks whose Wednesday is an exchange holiday (2 occurrences in the sample). Weeks with a holiday on some *other* weekday are kept: a Wed→Wed return spans exactly one calendar week regardless of how many trading sessions fall inside it.

We start with equities only, because their invariants are the simplest to construct (bonds and derivatives need the machinery of Ch. 3.1).

In [11]:
asset_symbol_list = ['DFUS', 'DFAI', 'DFAE']
start_date = date(year=2021, month=6, day=15)
end_date = date.today()
price_fetcher_factory = price.PriceFetcherFactory(adjust_prices=True)
price_fetcher = price_fetcher_factory.build_price_fetcher(
    asset_type=dt.AssetType.EQUITY,
    exchange=dt.Exchange.NYSE,
)

prices_dict = {}
weekly_returns_dict = {}
for asset_symbol in asset_symbol_list:
    price_history = price_fetcher.fetch_symbol(symbol=asset_symbol, start=start_date, end=end_date)
    prices_dict[asset_symbol] = pd.Series(
        {observation.date: observation.close for observation in price_history.closes}
    )
    weekly_returns = returns.convert_history_prices_to_weekly_returns(price_history)
    weekly_returns_dict[weekly_returns.symbol] = pd.Series(
        {r.date: r.equity_return for r in weekly_returns.equity_returns}
    )

# Align each column on its own dates: pandas outer-joins the per-symbol indices,
# so assets with differing trading calendars line up automatically.
assets_prices = pd.DataFrame(prices_dict)
assets_returns = pd.DataFrame(weekly_returns_dict)
print(assets_returns.head())

                DFUS      DFAI      DFAE
2021-06-23  0.005433 -0.020999  0.001914
2021-06-30  0.013287 -0.002051  0.011031
2021-07-07  0.011918  0.003723 -0.018912
2021-07-14  0.000084  0.004117 -0.001377
2021-07-21  0.000105 -0.012643 -0.013875


## Estimation
one-week distribution (Meucci Ch. 4)

Estimation step: the invariant distribution is summarized by the sample mean vector $\hat\mu$ and the sample covariance matrix $\hat\Sigma$ (with Bessel's correction, `ddof=1`).

Working assumption for everything downstream: $X \sim N(\hat\mu, \hat\Sigma)$, i.i.d. across weeks. With ≈266 weekly observations and only 3 assets, plain sample moments are an acceptable starting point. The histogram is a visual sanity check — the fat tails it reveals are deliberately ignored for now (the scenario framework of step 4 can later accommodate any invariant distribution).

Worth noting: **all statistical error of the entire pipeline lives in $\hat\mu$ and $\hat\Sigma$**. Every quantity computed below is deterministic arithmetic on these two estimates.

In [12]:
fig = px.histogram(assets_returns, x='DFUS', title='DFUS weekly log returns')
fig.show()

fig = px.histogram(assets_returns, x='DFAI', title='DFAI weekly log returns')
fig.show()

fig = px.histogram(assets_returns, x='DFAE', title='DFAE weekly log returns')
fig.show()

data = assets_returns.to_numpy()
# 1. Calculate the sample mean vector (for each column)
inv_mean_vector = np.mean(data, axis=0).reshape((3, 1))

# 2. Calculate the sample variance-covariance matrix (ddof=1 for sample/Bessel's correction)
inv_cov_matrix = np.cov(data, rowvar=False, ddof=1)

print("Sample Mean Vector:")
print(inv_mean_vector)

print("\nSample Variance-Covariance Matrix:")
print(inv_cov_matrix)

Sample Mean Vector:
[[0.00237028]
 [0.00187938]
 [0.00159192]]

Sample Variance-Covariance Matrix:
[[0.00050419 0.00037367 0.00039385]
 [0.00037367 0.00044079 0.00040205]
 [0.00039385 0.00040205 0.00056831]]


## Projection
to the investment horizon (Meucci Ch. 3.2)

Log returns add across time, so i.i.d. invariants project to a horizon $\tau$ by simple scaling (the "square-root rule" for volatility):

$$\mu_\tau = \frac{\tau}{\tilde\tau}\,\hat\mu, \qquad \Sigma_\tau = \frac{\tau}{\tilde\tau}\,\hat\Sigma,$$

where $\tilde\tau$ is the estimation interval. Here the horizon equals the estimation interval (1 week), so the factor is 1 — this cell exists so that the horizon can be changed in a single place.

In [13]:
tau_projected = 1 / 52 
tau_in_estimation = 1 / 52
projected_mean_vector = tau_projected / tau_in_estimation * inv_mean_vector
projected_cov_matrix = tau_projected / tau_in_estimation * inv_cov_matrix


## Market distribution (Meucci Ch. 3.3)

Horizon prices follow from the invariants by exact pricing: $P_{T+\tau} = p_T \circ e^{X_\tau}$. With normal invariants the market is therefore **lognormal**, not normal.

Meucci's Section 6.1 example nevertheless assumes $P_{T+\tau} \sim N(\xi, \Phi)$. This is a crude but deliberate approximation: normality of the market is precisely what makes every downstream object (CE, VaR, the optimal allocation itself) closed-form. To make the approximation as faithful as possible, $\xi$ and $\Phi$ are set to the **exact moments of the lognormal**:

$$\xi = p_T \circ e^{\mu_\tau + \frac{1}{2}\operatorname{diag}\Sigma_\tau}, \qquad \Phi = \xi\xi' \circ \left(e^{\Sigma_\tau} - 1\right).$$

Two classic traps hide in these formulas. First, $p \circ e^{\mu}$ alone is the *median* of the lognormal, not its mean — the extra $\frac{1}{2}\operatorname{diag}\Sigma$ is the Jensen correction for the convex map $e^x$ (symmetric log-shocks produce asymmetric dollar moves). Second, the price covariance entangles the means: in log space mean and covariance are clean and separate; the nonlinearity mixes them.

In [14]:
prices_vector = assets_prices.iloc[-1].to_numpy().reshape((3,1))
normal_in_usd_mean_vector = np.log(prices_vector) + projected_mean_vector
normal_in_usd_cov_matrix = projected_cov_matrix
market_mean_vector = np.exp(normal_in_usd_mean_vector + np.diag(normal_in_usd_cov_matrix).reshape((3,1)) / 2) 
market_cov_matrix = market_mean_vector @ market_mean_vector.transpose() * (np.exp(normal_in_usd_cov_matrix) - 1)
print("Sample Mean Vector:")
print(market_mean_vector)

print("\nSample Variance-Covariance Matrix:")
print(market_cov_matrix)

Sample Mean Vector:
[[84.5213595 ]
 [43.53130853]
 [40.06509652]]

Sample Variance-Covariance Matrix:
[[3.602797   1.37512616 1.3339898 ]
 [1.37512616 0.83546917 0.70134248]
 [1.3339898  0.70134248 0.91251832]]


## Utility function and index of satisfaction (Meucci Ch 5.4)

Exponential (CARA) utility over horizon wealth $\psi$:

$$u(\psi) = -e^{-\psi/\zeta}.$$

The risk tolerance $\zeta$ **must carry dollar units** — the argument of an exponential is a pure number. Interpretation: facing a fair $\pm z$ gamble, a CARA investor pays $\approx z^2/2\zeta$ to avoid it. Equivalently $\zeta = w/\lambda$, with $\lambda$ the dimensionless relative risk aversion (here $\lambda = 10$).

Why this utility: for CARA + normal market, the certainty-equivalent is *exactly* mean-variance (6.21), with no approximation:

$$CE(\alpha) = \xi'\alpha - \frac{\alpha'\Phi\alpha}{2\zeta}.$$

Note that CE is denominated in **horizon** dollars: it is the sure amount *at the horizon* equivalent to holding the risky portfolio. The definition $u(CE) = E[u(\Psi_\alpha)]$ never references time $T$, and there is no discounting anywhere in this model.

In [15]:
risk_tolerance = 10e6
def exp_utility_function(risk_tolerance: float, wealth_value: float) -> float:
    return - np.exp( - 1 / risk_tolerance * wealth_value)
wealth_value_list = np.arange(0, 100) * 1e6
utility_function = [exp_utility_function(risk_tolerance, x) for x in wealth_value_list]
fig = go.Figure()
fig.add_trace(go.Scatter(x=wealth_value_list, y=utility_function, mode='lines'))
fig.show()

## Portfolio optimization

The allocation problem (6.34): maximize CE subject to the full-investment budget $p_T'\alpha = w$ and a budget-at-risk constraint (risk measure $\le \gamma\,w$).

In [16]:
wealth = 100e6 
confidence_level = 0.95

### Analytical solution (6.39)

The closed-form maximizer of CE under the budget constraint alone:

$$\alpha^* = \zeta\,\Phi^{-1}\xi + \frac{w - \zeta\,p_T'\Phi^{-1}\xi}{p_T'\Phi^{-1}p_T}\,\Phi^{-1}p_T$$

— a *speculative* term (notably independent of wealth: a CARA signature) plus a *budget-absorbing* term. The risk constraint is not imposed here; we compute VaR (6.22) ex post and compare it with the risk budget. VaR is quoted in the **loss convention**: a positive number is a potential loss, $P(\text{weekly loss} > \mathrm{VaR}) = 5\%$.

In [17]:
solution = normal_market.analytical_solution(
    market_mean_vector=market_mean_vector,
    market_cov_matrix=market_cov_matrix,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
)

Allocations: [[ 749889.77724388]
 [1079655.53022003]
 [-252961.86135578]]
Certain equivalent: [[1.00025794e+08]]
VaR: [[3203014.61387714]]


### Convex optimization — VaR constraint (SOCP in cvxpy)

The same problem solved numerically, with three practical lessons baked into the formulation:

1. **Nondimensionalize.** Solvers want all quantities $O(1)$. Dollar-scale data (shares $\sim 10^5$, wealth $\sim 10^8$, quadratic coefficient $\sim 10^{-7}$) spans 14 orders of magnitude and breaks interior-point linear algebra. We therefore optimize portfolio *weights* $h_i = p_i\alpha_i/w$ with a unit budget; the natural data become gross returns $g = \xi / p_T$ and the per-dollar covariance $\Psi = \Phi / p_Tp_T'$.
2. **DCP rules.** `cvxpy` certifies convexity syntactically, so $\sqrt{h'\Psi h}$ must be written as `cp.norm(L.T @ h)` with $L$ the Cholesky factor of $\Psi$ — this *is* the second-order cone representation.
3. **Risk in the loss convention.** With $k_{VaR} = \sqrt{2}\,\operatorname{erf}^{-1}(2c-1) \approx 1.645$, VaR per unit of wealth reads $(1 - g'h) + k_{VaR}\,\sigma_{rel}$ and is constrained $\le \gamma_{VaR}$.

We set $\gamma_{VaR}$ to the VaR of the analytical allocation (6.39) — `value_at_risk / wealth`. The constraint therefore just binds at the closed-form optimum, so this program reproduces (6.39) to solver tolerance ($\sim 10^{-5}$ relative): a check that the SOCP formulation matches the analytical result. The printed budget dual is a shadow price — one extra dollar of budget buys that many dollars of certainty-equivalent.

In [18]:
gamma_var = (solution.value_at_risk / wealth).item()  # match the analytical VaR budget

normal_market.optimization_with_var_constrain(
    market_mean_vector=market_mean_vector,
    market_cov_matrix=market_cov_matrix,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_var,
)

Allocations: [[ 749477.61373809]
 [1079956.49902837]
 [-252419.94339495]]
CE: 100,025,794
budget shadow price (CE gained per extra dollar of budget): 0.9983


array([[ 749477.61373809],
       [1079956.49902837],
       [-252419.94339495]])

### Convex optimization — CVaR constraint

The same SOCP, now with the **CVaR** (normal expected-shortfall) risk measure: $k_{CVaR} = \dfrac{e^{-[\operatorname{erf}^{-1}(2c-1)]^2}}{(1-c)\sqrt{2\pi}} \approx 2.063$, so CVaR per unit of wealth reads $(1 - g'h) + k_{CVaR}\,\sigma_{rel}$ and is constrained $\le \gamma_{CVaR} = 4.1\%$ of wealth.

At equal confidence CVaR $>$ VaR always. Because the CVaR budget binds, the optimum legitimately differs a little from (6.39) — the closed form knows nothing about the risk constraint.

In [19]:
gamma_cvar = 0.041

normal_market.optimization_with_cvar_constrain(
    market_mean_vector=market_mean_vector,
    market_cov_matrix=market_cov_matrix,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
)

Allocations: [[ 749601.69269144]
 [1079865.89448204]
 [-252583.08404273]]
CE: 100,025,794
budget shadow price (CE gained per extra dollar of budget): 0.9981


array([[ 749601.69269144],
       [1079865.89448204],
       [-252583.08404273]])